# Step 4 - Model Development

**Baseline: SARIMAX with exogenous regressors.**

Cleaned dataset only - no engineered features.

Metrics are MAPE and RMSE per the brief, with WAPE alongside because MAPE is
undefined on zero-consumption days (12.2% of rows). Selection is on the validation
split; the test period is not touched.

In [1]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from mig_cement.config import settings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

TRAIN_END, VAL_END = "2024-06-30", "2024-09-30"
TARGET = "y"

## 1. Load the cleaned dataset

In [2]:
clean = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")
clean["date"] = pd.to_datetime(clean["date"])
clean = clean.sort_values(["site_id", "date"]).reset_index(drop=True)

print("shape:", clean.shape)
print("sites:", clean.site_id.nunique(), "| dates:", clean.date.nunique())
print("range:", clean.date.min().date(), "->", clean.date.max().date())

shape: (32880, 22)
sites: 30 | dates: 1096
range: 2022-01-01 -> 2024-12-31


## 2. Handle NaNs

Rows with NaN are removed, scoped to the columns this model uses.

A blanket `dropna()` would be a trap: `cover_days` is NaN exactly where
`consumed_tonnes == 0`, so it silently deletes every zero-consumption day - the
rain-blocked pours and stockouts, which are the hard cases.

In [3]:
print("NaN counts:")
print(clean.isna().sum()[lambda s: s > 0].to_string())
print("\nblanket dropna would give:", clean.dropna().shape,
      f"({100*(1-len(clean.dropna())/len(clean)):.1f}% lost)")
print("  zero-y rows before:", int((clean[TARGET] == 0).sum()),
      "| after:", int((clean.dropna()[TARGET] == 0).sum()))

NaN counts:
cover_days    4003

blanket dropna would give: (28877, 22) (12.2% lost)
  zero-y rows before: 4003 | after: 0


In [4]:
EXOG = ["planned_pour_tonnes", "rain_mm", "avg_temp_c", "opening_inventory_tonnes"]

before = len(clean)
clean = clean.dropna(subset=[TARGET] + EXOG).reset_index(drop=True)
print(f"rows: {before:,} -> {len(clean):,}")
print(f"zero-y rows retained: {int((clean[TARGET] == 0).sum()):,} "
      f"({(clean[TARGET] == 0).mean():.1%})")

rows: 32,880 -> 32,880
zero-y rows retained: 4,003 (12.2%)


### Why these four regressors

Of the 22 columns in the cleaned panel, most cannot be used:

- **target-derived / leaky**: `consumed_tonnes`, `served_tonnes`,
  `closing_inventory_tonnes`, `cover_days`, `silo_utilisation`, `was_constrained`,
  `unmet_tonnes`, `induced_shortfall`
- **not knowable at forecast time**: `deliveries_tonnes`, `received_tonnes`,
  `rejected_delivery_tonnes`
- **constant within each site**: `silo_capacity`, `region`, `behavior` - models are
  fitted per site, so these have no within-series variance and are collinear with
  the intercept
- **keys**: `date`, `site_id`, `cement_type`

## 3. Train / validation / test split

Chronological. The test period is held back.

In [5]:
d = clean["date"]
train = clean[d <= TRAIN_END]
val = clean[(d > TRAIN_END) & (d <= VAL_END)]
test = clean[d > VAL_END]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:6s} {len(part):6,} rows  {part.date.min().date()} -> {part.date.max().date()}")

train  27,360 rows  2022-01-01 -> 2024-06-30
val     2,760 rows  2024-07-01 -> 2024-09-30
test    2,760 rows  2024-10-01 -> 2024-12-31


## 4. Model configuration

`d = 0` because the series are stationary. `seasonal_order = (0,0,0,0)` because
Step 3 tested weekly, monthly and annual seasonality per region against a shuffled
null and found none - seasonal terms would fit noise.

In [6]:
adf = pd.Series({s: adfuller(g[TARGET])[1] for s, g in clean.groupby("site_id")})
print(f"ADF p-values across {len(adf)} sites: max = {adf.max():.2e}")
print(f"sites rejecting a unit root at 1%: {(adf < 0.01).sum()} / {len(adf)}")
print("\n-> d = 0")

ADF p-values across 30 sites: max = 3.61e-17
sites rejecting a unit root at 1%: 30 / 30

-> d = 0


In [7]:
# order chosen by mean AIC across a sample of sites
GRID = [(1, 0, 0), (0, 0, 1), (1, 0, 1), (2, 0, 1), (2, 0, 2)]
aic = {}
for o in GRID:
    scores = []
    for site in sorted(train.site_id.unique())[:5]:
        g = train[train.site_id == site].set_index("date")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic[str(o)] = np.mean(scores)

aic = pd.Series(aic).sort_values()
print(aic.round(1).to_string())
ORDER = (2, 0, 2)
print("\nselected:", ORDER)

(2, 0, 2)    4979.0
(2, 0, 1)    4984.3
(1, 0, 1)    4984.7
(1, 0, 0)    4985.9
(0, 0, 1)    4986.2

selected: (2, 0, 2)


## 5. Train the model

Fitted on one site first, in the plainest form.

In [8]:
SITE = "SITE_001"

y_train = train[train.site_id == SITE].set_index("date")[TARGET]
x_train = train[train.site_id == SITE].set_index("date")[EXOG]
y_val = val[val.site_id == SITE].set_index("date")[TARGET]
x_val = val[val.site_id == SITE].set_index("date")[EXOG]

print(f"{SITE}: train {y_train.shape[0]} rows, val {y_val.shape[0]} rows, "
      f"{x_train.shape[1]} exogenous regressors")

SITE_001: train 912 rows, val 92 rows, 4 exogenous regressors


In [9]:
model = SARIMAX(
    y_train,
    exog=x_train,
    order=ORDER,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results = model.fit(disp=False)
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                  912
Model:               SARIMAX(2, 0, 2)   Log Likelihood               -3488.121
Date:                Thu, 06 Aug 2026   AIC                           6994.243
Time:                        13:53:24   BIC                           7037.584
Sample:                    01-01-2022   HQIC                          7010.789
                         - 06-30-2024                                         
Covariance Type:                  opg                                         
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
planned_pour_tonnes          0.6497      0.017     37.581      0.000       0.616       0.684
rain_mm                     -0.7636      0.054    -14.242      0.000      -0.869      -0.659
avg_temp_c                   0.1877      0.047      3.964      0.000       0.095       0.281
opening_inventory_tonnes     0.2729      0.020     13.435      0.000       0.233       0.313
ar.L1                        0.5999      1.215      0.494      0.622      -1.782       2.982
ar.L2                        0.2454      1.007      0.244      0.808      -1.729       2.220
ma.L1                       -0.5355      1.210     -0.443      0.658      -2.906       1.835
ma.L2                       -0.2605      0.941     -0.277      0.782      -2.104       1.583
sigma2                     123.0785      7.080     17.384      0.000     109.202     136.955
===================================================================================
Ljung-Box (L1) (Q):                   0.03   Jarque-Bera (JB):                60.37
Prob(Q):                              0.86   Prob(JB):                         0.00
Heteroskedasticity (H):               1.03   Skew:                            -0.63
Prob(H) (two-sided):                  0.82   Kurtosis:                         2.96
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

`enforce_stationarity=True` is deliberate. With it set to `False`, an unstable
AR root produced forecasts that diverged across the 92-day validation window -
one site reached an RMSE of 1.96e20. The series are stationary, so the constraint
costs nothing.

## 6. Predict

`y_test` / `X_test` below are the **validation** window (Jul-Sep 2024).
Oct-Dec 2024 stays held back and is not touched in this notebook.

In [10]:
y_test = y_val
X_test = x_val

y_pred = results.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
y_pred = y_pred.clip(lower=0)
y_pred

2024-07-01    29.360824
2024-07-02    22.726965
2024-07-03    36.482038
2024-07-04    32.125625
2024-07-05    39.117138
                ...    
2024-09-26    36.899913
2024-09-27    34.792872
2024-09-28    29.244113
2024-09-29    39.149126
2024-09-30    37.968296
Freq: D, Name: predicted_mean, Length: 92, dtype: float64

## 7. Metrics

`sklearn.metrics.mean_absolute_percentage_error` returns a **fraction, not a
percentage** - a returned value of 15 means 1500%, not 15%.

It is also undefined when the actual is zero. 12.2% of site-days have no pour, and
on those rows sklearn divides by a tiny epsilon, so a handful of rows can dominate
the whole average. MAPE is therefore computed on non-zero actuals only, with the
raw sklearn figure shown alongside so the gap is visible.

In [11]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

nz = y_test != 0

mape_raw = mean_absolute_percentage_error(y_test, y_pred)
mape_nonzero = mean_absolute_percentage_error(y_test[nz], y_pred[nz])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"{SITE}")
print(f"  zero-actual days      {int((~nz).sum())} of {len(y_test)}")
print(f"  MAPE (sklearn raw)    {mape_raw:.4f}  = {mape_raw*100:,.0f}%")
print(f"  MAPE (non-zero only)  {mape_nonzero:.4f}  = {mape_nonzero*100:.1f}%")
print(f"  RMSE                  {rmse:.3f} t")

SITE_001
  zero-actual days      16 of 92
  MAPE (sklearn raw)    9531870149063316.0000  = 953,187,014,906,331,648%
  MAPE (non-zero only)  0.2595  = 26.0%
  RMSE                  10.755 t


## 8. Fit all 30 sites and predict

In [12]:
models, preds = {}, []

for site, g_tr in train.groupby("site_id"):
    g_tr = g_tr.set_index("date")
    g_te = val[val.site_id == site].sort_values("date").set_index("date")

    y_tr, X_tr = g_tr[TARGET], g_tr[EXOG]
    y_te, X_te = g_te[TARGET], g_te[EXOG]

    try:
        res = SARIMAX(
            y_tr,
            exog=X_tr,
            order=ORDER,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
        pred = res.predict(start=y_te.index[0], end=y_te.index[-1], exog=X_te).clip(lower=0)
    except Exception:
        res, pred = None, pd.Series(np.nan, index=y_te.index)

    models[site] = res
    preds.append(pd.DataFrame({"date": y_te.index, "site_id": site,
                               "actual": y_te.values, "pred": pred.values}))

fc = pd.concat(preds, ignore_index=True).dropna(subset=["pred"])
converged = sum(m is not None for m in models.values())
print(f"sites: {len(models)} | converged: {converged} | failed: {len(models) - converged}")
print(f"{len(fc):,} predictions | {fc.date.nunique()} dates x {fc.site_id.nunique()} sites")

sites: 30 | converged: 30 | failed: 0
2,760 predictions | 92 dates x 30 sites


## 9. Daily error, per site

One row per site-day: absolute error, squared error, and percentage error where the
actual is non-zero.

In [13]:
fc["abs_err"] = (fc.actual - fc.pred).abs()
fc["sq_err"] = (fc.actual - fc.pred) ** 2
fc["pct_err"] = np.where(fc.actual != 0, fc.abs_err / fc.actual, np.nan)

print(f"{len(fc):,} site-days | {fc.pct_err.isna().sum():,} with zero actual (no MAPE)")
fc.head(10).round(3)

2,760 site-days | 331 with zero actual (no MAPE)


,date,site_id,actual,pred,abs_err,sq_err,pct_err
0,2024-07-01,SITE_001,26.75,29.361,2.611,6.816,0.098
1,2024-07-02,SITE_001,22.51,22.727,0.217,0.047,0.010
2,2024-07-03,SITE_001,15.48,36.482,21.002,441.086,1.357
3,2024-07-04,SITE_001,44.07,32.126,11.944,142.668,0.271
4,2024-07-05,SITE_001,40.07,39.117,0.953,0.908,0.024
5,2024-07-06,SITE_001,30.06,17.259,12.801,163.857,0.426
6,2024-07-07,SITE_001,0.00,0.000,0.000,0.000,NaN
7,2024-07-08,SITE_001,35.45,30.025,5.425,29.427,0.153
8,2024-07-09,SITE_001,38.26,32.267,5.993,35.917,0.157
9,2024-07-10,SITE_001,14.56,33.081,18.521,343.011,1.272


In [14]:
per_site = pd.DataFrame({
    "n_days": fc.groupby("site_id").size(),
    "zero_days": fc.groupby("site_id").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc.groupby("site_id").actual.mean(),
    "MAPE": fc.groupby("site_id").pct_err.mean(),
    "RMSE": fc.groupby("site_id").sq_err.mean() ** 0.5,
}).sort_values("MAPE", ascending=False)

print(f"MAPE across sites: best {per_site.MAPE.min():.1%} | "
      f"median {per_site.MAPE.median():.1%} | worst {per_site.MAPE.max():.1%}")
per_site.round(4)

MAPE across sites: best 6.3% | median 25.0% | worst 42.7%


,n_days,zero_days,mean_actual,MAPE,RMSE
site_id,,,,,
SITE_017,92,8,28.1239,0.4273,12.5276
SITE_030,92,7,29.1316,0.3880,10.4852
SITE_025,92,7,30.8111,0.3872,11.8301
SITE_021,92,9,28.7862,0.3827,10.2458
SITE_010,92,4,30.0573,0.3704,10.6331
SITE_018,92,6,29.5473,0.3676,10.6119
SITE_022,92,8,29.5374,0.3621,10.6780
SITE_008,92,13,28.3165,0.3610,10.8089
SITE_006,92,19,25.7420,0.3145,10.8764


## 10. Daily error, per calendar date across all sites

In [15]:
per_day = pd.DataFrame({
    "n_sites": fc.groupby("date").size(),
    "zero_sites": fc.groupby("date").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc.groupby("date").actual.mean(),
    "mean_pred": fc.groupby("date").pred.mean(),
    "MAPE": fc.groupby("date").pct_err.mean(),
    "RMSE": fc.groupby("date").sq_err.mean() ** 0.5,
})

print(f"{len(per_day)} days | MAPE best {per_day.MAPE.min():.1%} | "
      f"median {per_day.MAPE.median():.1%} | worst {per_day.MAPE.max():.1%}")
per_day.round(4)

92 days | MAPE best 12.7% | median 21.8% | worst 42.6%


,n_sites,zero_sites,mean_actual,mean_pred,MAPE,RMSE
date,,,,,,
2024-07-01,30,4,25.9203,24.6218,0.1827,7.8787
2024-07-02,30,5,21.3010,22.6588,0.2189,7.2019
2024-07-03,30,4,24.4493,22.5750,0.1624,7.1432
2024-07-04,30,3,26.7497,23.9993,0.1565,7.4644
2024-07-05,30,4,22.8220,21.0851,0.2013,8.5446
...,...,...,...,...,...,...
2024-09-26,30,1,25.2933,23.3527,0.2449,8.4230
2024-09-27,30,4,18.4490,20.0368,0.3022,7.3995
2024-09-28,30,7,16.0650,20.5048,0.3231,10.4250


In [16]:
per_day_h = per_day.reset_index()
per_day_h["horizon_day"] = np.arange(1, len(per_day_h) + 1)
per_day_h["week"] = ((per_day_h.horizon_day - 1) // 7) + 1

by_week = per_day_h.groupby("week").agg(
    days=("horizon_day", "size"), MAPE=("MAPE", "mean"), RMSE=("RMSE", "mean")).head(8)
print("error by forecast week (the 8-week horizon in the brief):")
by_week.round(4)

error by forecast week (the 8-week horizon in the brief):


,days,MAPE,RMSE
week,,,
1,7,0.1810,7.3412
2,7,0.2165,7.7288
3,7,0.2026,7.7241
4,7,0.2224,9.0557
5,7,0.2307,8.7417
6,7,0.2286,8.9272
7,7,0.2780,9.9041
8,7,0.2117,8.2263


## 11. Overall, against baselines

In [17]:
def score(y_true, y_pred_):
    y_true, y_pred_ = np.asarray(y_true, float), np.asarray(y_pred_, float)
    nzm = y_true != 0
    return {
        "MAPE": mean_absolute_percentage_error(y_true[nzm], y_pred_[nzm]),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred_)),
        "WAPE": np.abs(y_true - y_pred_).sum() / np.abs(y_true).sum(),
        "MAE": np.abs(y_true - y_pred_).mean(),
        "bias": (y_pred_ - y_true).mean(),
    }


rows = [
    {**score(val[TARGET], val["planned_pour_tonnes"]), "model": "baseline: planned_pour"},
    {**score(val[TARGET], np.full(len(val), train[TARGET].mean())), "model": "baseline: train mean"},
    {**score(fc.actual, fc.pred), "model": "SARIMAX (cleaned data)"},
]
results_tbl = pd.DataFrame(rows).set_index("model")[["MAPE", "RMSE", "WAPE", "MAE", "bias"]]
results_tbl.round(4)

,MAPE,RMSE,WAPE,MAE,bias
model,,,,,
baseline: planned_pour,0.3372,14.4142,0.3189,7.4622,7.4622
baseline: train mean,0.7077,16.6324,0.6094,14.2586,0.3923
SARIMAX (cleaned data),0.2227,8.6136,0.2464,5.7654,-0.0304


---

# Experiment 2 - Weekly Aggregation

Same cleaned dataset, same regressors, same model. The only change is the grain:
each site-day is aggregated to a site-week.

Consumption and planned pour are **summed**, weather is **averaged**, and opening
inventory takes the **first** value of the week.

In [18]:
AGG = {
    "y": "sum",
    "planned_pour_tonnes": "sum",
    "rain_mm": "mean",
    "avg_temp_c": "mean",
    "opening_inventory_tonnes": "first",
}

weekly = (clean.set_index("date")
               .groupby("site_id")
               .resample("W")
               .agg(AGG)
               .reset_index()
               .dropna(subset=["y"])
               .sort_values(["site_id", "date"])
               .reset_index(drop=True))

print("daily :", clean.shape, "| mean y", round(clean.y.mean(), 2), "t",
      "| zero rows", f"{(clean.y == 0).mean():.1%}")
print("weekly:", weekly.shape, "| mean y", round(weekly.y.mean(), 2), "t",
      "| zero rows", f"{(weekly.y == 0).mean():.1%}")
weekly.head()

daily : (32880, 22) | mean y 23.72 t | zero rows 12.2%
weekly: (4740, 7) | mean y 164.54 t | zero rows 0.0%


,site_id,date,y,planned_pour_tonnes,rain_mm,avg_temp_c,opening_inventory_tonnes
0,SITE_001,2022-01-02,79.80,88.44,3.315000,5.590000,52.56
1,SITE_001,2022-01-09,208.96,234.47,2.734286,13.060000,38.56
2,SITE_001,2022-01-16,269.66,286.58,3.372857,11.337143,34.38
3,SITE_001,2022-01-23,235.01,346.98,3.258571,12.935714,4.95
4,SITE_001,2022-01-30,235.11,341.55,7.021429,12.470000,7.22


Aggregation removes the zero-consumption problem entirely: no site goes a full
week without pouring, so MAPE becomes well-defined on every row.

## Split

In [19]:
dw = weekly["date"]
train_w = weekly[dw <= TRAIN_END]
val_w = weekly[(dw > TRAIN_END) & (dw <= VAL_END)]
test_w = weekly[dw > VAL_END]

for name, part in [("train", train_w), ("val", val_w), ("test", test_w)]:
    print(f"{name:6s} {len(part):5,} rows  {part.date.min().date()} -> {part.date.max().date()}"
          f"  ({part.groupby('site_id').size().mean():.0f} weeks per site)")

train  3,930 rows  2022-01-02 -> 2024-06-30  (131 weeks per site)
val      390 rows  2024-07-07 -> 2024-09-29  (13 weeks per site)
test     420 rows  2024-10-06 -> 2025-01-05  (14 weeks per site)


## Order selection

In [20]:
aic_w = {}
for o in GRID:
    scores = []
    for site in sorted(train_w.site_id.unique())[:5]:
        g = train_w[train_w.site_id == site].set_index("date").asfreq("W")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic_w[str(o)] = np.mean(scores) if scores else np.nan

aic_w = pd.Series(aic_w).sort_values()
print(aic_w.round(1).to_string())
ORDER_W = eval(aic_w.index[0])
print("\nselected:", ORDER_W)

(2, 0, 2)     998.8
(1, 0, 1)    1000.5
(1, 0, 0)    1002.9
(0, 0, 1)    1005.2
(2, 0, 1)    1213.7

selected: (2, 0, 2)


## Train the model

In [21]:
y_train_w = train_w[train_w.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_train_w = train_w[train_w.site_id == SITE].set_index("date").asfreq("W")[EXOG]
y_test_w = val_w[val_w.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_test_w = val_w[val_w.site_id == SITE].set_index("date").asfreq("W")[EXOG]

print(f"{SITE}: train {len(y_train_w)} weeks, val {len(y_test_w)} weeks")

SITE_001: train 131 weeks, val 13 weeks


In [22]:
model_w = SARIMAX(
    y_train_w,
    exog=x_train_w,
    order=ORDER_W,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results_w = model_w.fit(disp=False)
results_w.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                  131
Model:               SARIMAX(2, 0, 2)   Log Likelihood                -636.481
Date:                Thu, 06 Aug 2026   AIC                           1290.962
Time:                        13:53:39   BIC                           1316.838
Sample:                    01-02-2022   HQIC                          1301.477
                         - 06-30-2024                                         
Covariance Type:                  opg                                         
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
planned_pour_tonnes          0.6400      0.027     23.818      0.000       0.587       0.693
rain_mm                     -0.2891      1.271     -0.227      0.820      -2.781       2.203
avg_temp_c                   0.4710      0.385      1.224      0.221      -0.283       1.225
opening_inventory_tonnes     1.0373      0.119      8.734      0.000       0.805       1.270
ar.L1                        0.8749      0.073     11.989      0.000       0.732       1.018
ar.L2                       -0.8081      0.082     -9.840      0.000      -0.969      -0.647
ma.L1                       -1.0544      0.481     -2.191      0.028      -1.998      -0.111
ma.L2                        0.9977      0.917      1.088      0.276      -0.799       2.794
sigma2                     935.5812    818.492      1.143      0.253    -668.634    2539.797
===================================================================================
Ljung-Box (L1) (Q):                   0.15   Jarque-Bera (JB):                 0.90
Prob(Q):                              0.70   Prob(JB):                         0.64
Heteroskedasticity (H):               1.04   Skew:                            -0.18
Prob(H) (two-sided):                  0.90   Kurtosis:                         2.83
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

## Predict

In [23]:
y_pred_w = results_w.predict(start=y_test_w.index[0], end=y_test_w.index[-1], exog=x_test_w)
y_pred_w = y_pred_w.clip(lower=0)

print(f"{SITE}")
print(f"  MAPE {mean_absolute_percentage_error(y_test_w, y_pred_w):.4f}"
      f"  = {mean_absolute_percentage_error(y_test_w, y_pred_w)*100:.1f}%")
print(f"  RMSE {np.sqrt(mean_squared_error(y_test_w, y_pred_w)):.3f} t")
y_pred_w

SITE_001
  MAPE 0.1325  = 13.2%
  RMSE 32.247 t


2024-07-07    179.735284
2024-07-14    235.354548
2024-07-21    159.388594
2024-07-28    226.212757
2024-08-04    242.136609
2024-08-11    257.590550
2024-08-18    219.556083
2024-08-25    175.687086
2024-09-01    215.572245
2024-09-08    183.751364
2024-09-15    239.724512
2024-09-22    146.019095
2024-09-29    281.656141
Freq: W-SUN, Name: predicted_mean, dtype: float64

## Fit all 30 sites

In [24]:
models_w, preds_w = {}, []

for site, g_tr in train_w.groupby("site_id"):
    g_tr = g_tr.set_index("date").asfreq("W")
    g_te = val_w[val_w.site_id == site].sort_values("date").set_index("date").asfreq("W")

    try:
        res = SARIMAX(
            g_tr[TARGET],
            exog=g_tr[EXOG],
            order=ORDER_W,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
        pred = res.predict(start=g_te.index[0], end=g_te.index[-1],
                           exog=g_te[EXOG]).clip(lower=0)
    except Exception:
        res, pred = None, pd.Series(np.nan, index=g_te.index)

    models_w[site] = res
    preds_w.append(pd.DataFrame({"date": g_te.index, "site_id": site,
                                 "actual": g_te[TARGET].values, "pred": pred.values}))

fc_w = pd.concat(preds_w, ignore_index=True).dropna(subset=["pred"])
converged_w = sum(m is not None for m in models_w.values())
print(f"sites: {len(models_w)} | converged: {converged_w} | failed: {len(models_w) - converged_w}")
print(f"{len(fc_w):,} predictions | {fc_w.date.nunique()} weeks x {fc_w.site_id.nunique()} sites")

sites: 30 | converged: 30 | failed: 0
390 predictions | 13 weeks x 30 sites


## Weekly error, per calendar week across all sites

In [25]:
fc_w["abs_err"] = (fc_w.actual - fc_w.pred).abs()
fc_w["sq_err"] = (fc_w.actual - fc_w.pred) ** 2
fc_w["pct_err"] = np.where(fc_w.actual != 0, fc_w.abs_err / fc_w.actual, np.nan)

per_week = pd.DataFrame({
    "n_sites": fc_w.groupby("date").size(),
    "zero_sites": fc_w.groupby("date").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc_w.groupby("date").actual.mean(),
    "mean_pred": fc_w.groupby("date").pred.mean(),
    "MAPE": fc_w.groupby("date").pct_err.mean(),
    "RMSE": fc_w.groupby("date").sq_err.mean() ** 0.5,
})

print(f"{len(per_week)} weeks | MAPE best {per_week.MAPE.min():.1%} | "
      f"median {per_week.MAPE.median():.1%} | worst {per_week.MAPE.max():.1%}")
per_week.round(4)

13 weeks | MAPE best 7.5% | median 10.6% | worst 17.6%


,n_sites,zero_sites,mean_actual,mean_pred,MAPE,RMSE
date,,,,,,
2024-07-07,30,0,174.7333,166.1398,0.0793,24.4392
2024-07-14,30,0,175.2223,170.5361,0.1065,28.6847
2024-07-21,30,0,169.4383,159.6357,0.0805,19.7590
2024-07-28,30,0,157.9543,165.2941,0.1076,26.2291
2024-08-04,30,0,160.8960,161.9213,0.0748,19.6836
2024-08-11,30,0,162.0863,165.5905,0.0945,25.6836
2024-08-18,30,0,164.8863,178.0080,0.1627,45.6210
2024-08-25,30,0,163.2863,159.5475,0.0768,22.2304
2024-09-01,30,0,159.5650,163.9726,0.1755,31.6962


## Weekly error, per site

In [26]:
per_site_w = pd.DataFrame({
    "n_weeks": fc_w.groupby("site_id").size(),
    "mean_actual": fc_w.groupby("site_id").actual.mean(),
    "MAPE": fc_w.groupby("site_id").pct_err.mean(),
    "RMSE": fc_w.groupby("site_id").sq_err.mean() ** 0.5,
}).sort_values("MAPE", ascending=False)

print(f"MAPE across sites: best {per_site_w.MAPE.min():.1%} | "
      f"median {per_site_w.MAPE.median():.1%} | worst {per_site_w.MAPE.max():.1%}")
per_site_w.round(4)

MAPE across sites: best 2.3% | median 10.8% | worst 43.1%


,n_weeks,mean_actual,MAPE,RMSE
site_id,,,,
SITE_016,13,178.4569,0.4305,55.7825
SITE_008,13,197.0485,0.2278,46.8325
SITE_014,13,191.0900,0.1951,54.0722
SITE_006,13,180.2285,0.1862,35.8557
SITE_021,13,200.9362,0.1837,37.8284
SITE_025,13,215.1846,0.1429,38.2064
SITE_020,13,205.9700,0.1380,31.7925
SITE_026,13,229.2254,0.1332,34.7895
SITE_001,13,201.5823,0.1325,32.2470


## Daily vs weekly

In [27]:
rows_w = [
    {**score(val_w[TARGET], val_w["planned_pour_tonnes"]), "model": "planned_pour (weekly)"},
    {**score(fc_w.actual, fc_w.pred), "model": "SARIMAX (weekly)"},
]
comparison = pd.concat([results_tbl, pd.DataFrame(rows_w).set_index("model")[
    ["MAPE", "RMSE", "WAPE", "MAE", "bias"]]])
comparison.round(4)

,MAPE,RMSE,WAPE,MAE,bias
model,,,,,
baseline: planned_pour,0.3372,14.4142,0.3189,7.4622,7.4622
baseline: train mean,0.7077,16.6324,0.6094,14.2586,0.3923
SARIMAX (cleaned data),0.2227,8.6136,0.2464,5.7654,-0.0304
planned_pour (weekly),0.2908,74.1864,0.3177,52.1996,52.1996
SARIMAX (weekly),0.1108,28.4701,0.1143,18.7814,1.0420


**RMSE is not comparable across grains.** A weekly total is roughly seven times a
daily value, so its RMSE is larger by construction - that is arithmetic, not a
worse model. MAPE and WAPE are scale-relative and can be compared.

Aggregation also makes the problem mechanically easier: day-to-day noise cancels
when summed, and the 12.2% of zero-pour days disappear. A lower weekly MAPE is
therefore partly a real gain in usable accuracy and partly an easier question. The
figure that carries meaning is the **gap between SARIMAX and `planned_pour` at each
grain**, since both face the same conditions.

Weekly is also the grain MIG actually reorders on, which is the practical argument
for it regardless of the arithmetic.

## Notes

- Cleaned dataset only, no engineered features.
- `deliveries_tonnes`, `closing_inventory_tonnes` and `silo_capacity` are excluded
  from the regressors. The first two satisfy
  `consumed = opening + deliveries - closing` exactly, so including them lets the
  model reproduce the target instead of forecasting it; the third is constant
  within a site and makes the covariance matrix singular.
- MAPE is computed on non-zero actuals; the raw sklearn value is in section 7.
- Weather regressors use actual validation values, which flatters the result.
- The Oct-Dec 2024 test split is untouched.